# Estrazione della Sottorete Tematica

Estrae dal corpus normativo UE una **sottorete tematica** in due livelli:

1. **L1 — Seed manuali** (opzionale): atti fondamentali la cui appartenenza alla materia è documentata da fonti ufficiali
2. **L2 — Espansione EuroVoc**: tutti gli atti collegati ai concetti EuroVoc rilevanti per la materia

Gli archi interni vengono estratti per la visualizzazione ma non influenzano la selezione dei nodi.

## Modalità

| Modalità | Quando usarla | Come trova i concetti EuroVoc |
|---|---|---|
| `theme` | Esplorazione libera: scrivi la materia in inglese | Embedding semantico: confronta la query con tutti i 7.613 nomi EuroVoc, prende i più simili |
| `celex` | Hai già CELEX noti: parti dagli atti che conosci | Legge i tag EuroVoc già associati a quegli atti nel corpus |

**Nota**: la modalità `theme` richiede `sentence-transformers`. Prima esecuzione: scarica il modello (~420MB, una volta sola).

## 0. Configurazione

**Modifica solo questa cella.** Il resto del notebook gira in automatico.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
#  MODALITÀ  →  "theme" | "celex"
# ─────────────────────────────────────────────────────────────────────────────

MODE = "theme"


# ── MODE = "theme" ────────────────────────────────────────────────────────────
# Descrivi la materia in inglese. Più la descrizione è specifica e tecnica,
# migliore è la selezione EuroVoc.
# Esempi: "foreign direct investment screening"
#         "data protection personal data privacy"
#         "public procurement contracts"

THEME_QUERY        = "foreign direct investment screening"   # <- modifica qui
THEME_MATERIA_NAME = "fdi_screening"                         # nome cartella output
THEME_TOP_K        = 20     # quanti concetti EuroVoc più simili includere
                            # (10-25 per materie specifiche, 25-40 per materie ampie)
THEME_MIN_SIMILARITY = 0.25  # soglia minima similarità coseno (0-1)
                              # aumenta per sottorete più stringente


# ── MODE = "celex" ────────────────────────────────────────────────────────────
# Fornisci uno o più CELEX noti. I loro concetti EuroVoc definiscono il
# perimetro tematico per L2.

CELEX_SEEDS        = [
    "32019R0452",   # Reg. (UE) 2019/452 - FDI Screening Framework
    "32022L2557",   # Dir. (UE) 2022/2557 - CER
]
CELEX_MATERIA_NAME = "fdi_celex"   # nome cartella output


# ── Parametro comune a entrambe le modalità ───────────────────────────────────
# Numero minimo di concetti seed che un atto deve avere per entrare in L2.
# 1 = qualsiasi atto con almeno 1 concetto seed (sottorete ampia)
# 2 = solo atti con affinità tematica forte (sottorete più stringente)
MIN_SEED_CONCEPTS = 2

## 1. Import e Caricamento Dati

In [2]:
import pandas as pd
import numpy as np
import os
import re

proc_path = os.path.join('..', 'data', 'processed')
raw_path  = os.path.join('..', 'data', 'raw')

nodes       = pd.read_csv(os.path.join(proc_path, 'nodes_light.csv'))
edges       = pd.read_csv(os.path.join(proc_path, 'edges_enriched.csv'))
has_concept = pd.read_csv(os.path.join(proc_path, 'has_concept_enriched.csv'))
eurovoc     = pd.read_csv(os.path.join(raw_path,  'eurovoc_concept.csv'))

print("Corpus caricato:")
print(f"  Nodi:             {len(nodes):>7,}")
print(f"  Archi:            {len(edges):>7,}")
print(f"  Has-concept:      {len(has_concept):>7,}")
print(f"  Concetti EuroVoc: {len(eurovoc):>7,}")

MATERIA_NAME = THEME_MATERIA_NAME if MODE == "theme" else CELEX_MATERIA_NAME
output_path  = os.path.join('..', 'data', 'output', MATERIA_NAME)
os.makedirs(output_path, exist_ok=True)
print(f"\nModalità: {MODE.upper()}  |  Output: {os.path.abspath(output_path)}")

Corpus caricato:
  Nodi:              62,244
  Archi:            187,514
  Has-concept:      222,601
  Concetti EuroVoc:   7,613

Modalità: THEME  |  Output: c:\Users\claud\Documents\GitHub\eu-law-network-viz\data\output\fdi_screening


## 2. Identificazione Concetti EuroVoc

Cella chiave: identifica quali concetti EuroVoc rappresentano la materia.

- **`theme`**: embedding semantico della query vs tutti i nomi EuroVoc con `all-mpnet-base-v2`. Prende i `THEME_TOP_K` concetti con similarità coseno più alta sopra la soglia `THEME_MIN_SIMILARITY`. Il modello confronta il significato della query con quello di ogni nome EuroVoc nello spazio vettoriale — funziona indipendentemente dalla sovrapposizione letterale delle parole.
- **`celex`**: legge direttamente i tag EuroVoc degli atti seed dal corpus. Zero ambiguità: usa la classificazione ufficiale già assegnata da EUR-Lex.

In [3]:
if MODE == "theme":
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    print(f"Query: '{THEME_QUERY}'")
    print("Caricamento modello all-mpnet-base-v2 (prima esecuzione: ~420MB)...")
    # carica modello
    model = SentenceTransformer('all-mpnet-base-v2')

    eurovoc_names = eurovoc['name'].fillna('').tolist()  

    query_emb = model.encode([THEME_QUERY], show_progress_bar=False)

    EMBEDDINGS_CACHE = os.path.join(proc_path, 'eurovoc_embeddings.npy')
    if os.path.exists(EMBEDDINGS_CACHE):
        print("Cache trovata, carico embeddings EuroVoc...")
        concepts_emb = np.load(EMBEDDINGS_CACHE)
    else:
        print("Prima esecuzione: calcolo embeddings (~1-2 min)...")
        concepts_emb = model.encode(eurovoc_names, show_progress_bar=True, batch_size=256)
        np.save(EMBEDDINGS_CACHE, concepts_emb)
        print(f"Salvati in {EMBEDDINGS_CACHE}")

    # similarità coseno
    similarities = cosine_similarity(query_emb, concepts_emb)[0]
    eurovoc = eurovoc.copy()
    eurovoc['similarity'] = similarities

    # Filtra per soglia + top-k
    above_threshold = eurovoc[eurovoc['similarity'] >= THEME_MIN_SIMILARITY]
    seed_concepts_filtered = above_threshold.nlargest(THEME_TOP_K, 'similarity').copy()

    # Seed manuali: nessuno in modalità theme
    seed_nodes_known = nodes.iloc[0:0].copy()

    print(f"\nConcetti EuroVoc sopra soglia {THEME_MIN_SIMILARITY}: {len(above_threshold)}")
    print(f"Concetti selezionati (top-{THEME_TOP_K}): {len(seed_concepts_filtered)}")
    print()
    print("Concetti selezionati per similarità:")
    for _, row in seed_concepts_filtered.iterrows():
        print(f"  [{row['similarity']:.3f}] {row['name']}")


elif MODE == "celex":
    seed_nodes_known = nodes[nodes['celex'].isin(CELEX_SEEDS)].copy()
    found   = set(seed_nodes_known['celex'])
    missing = [c for c in CELEX_SEEDS if c not in found]

    print(f"Seed L1 trovati: {len(seed_nodes_known)} su {len(CELEX_SEEDS)}")
    for c in CELEX_SEEDS:
        status = "OK" if c in found else "MANCANTE"
        print(f"  [{status}] {c}")
    if missing:
        print(f"  Mancanti: {missing}")

    if seed_nodes_known.empty:
        raise ValueError("Nessun atto seed trovato. Verifica i CELEX in CELEX_SEEDS.")

    seed_node_ids          = set(seed_nodes_known['id'])
    concept_ids_from_seeds = set(
        has_concept[has_concept[':START_ID'].isin(seed_node_ids)][':END_ID']
    )
    seed_concepts_filtered = eurovoc[
        eurovoc['id:ID'].isin(concept_ids_from_seeds)
    ].copy()

    print(f"\nConcetti EuroVoc dai seed: {len(seed_concepts_filtered)}")
    for name in seed_concepts_filtered['name'].tolist():
        print(f"  - {name}")


else:
    raise ValueError(f"MODE non valido: '{MODE}'. Scegli 'theme' o 'celex'.")

Query: 'foreign direct investment screening'
Caricamento modello all-mpnet-base-v2 (prima esecuzione: ~420MB)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cache trovata, carico embeddings EuroVoc...

Concetti EuroVoc sopra soglia 0.25: 417
Concetti selezionati (top-20): 20

Concetti selezionati per similarità:
  [0.660] foreign investment
  [0.614] international investment
  [0.574] direct investment
  [0.547] investment abroad
  [0.534] Multilateral Investment Guarantee Agency
  [0.516] foreign capital
  [0.503] foreign enterprise
  [0.498] surveillance concerning imports
  [0.482] regional investment
  [0.473] international protection
  [0.471] international finance
  [0.466] regulation of investments
  [0.460] International Centre for Settlement of Investment Disputes
  [0.454] investment policy
  [0.450] multinational enterprise
  [0.441] foreign market
  [0.439] trade regulations
  [0.434] foreign trade
  [0.432] restrictive trade practice
  [0.430] export financing


## 3. Recupero Atti L2

Recupera tutti gli atti del corpus che hanno almeno `MIN_SEED_CONCEPTS` dei concetti EuroVoc selezionati.

La tabella di distribuzione permette di scegliere la soglia guardando il tradeoff tra ampiezza e stringenza: ogni riga mostra quanti atti sopravvivono alzando la soglia di 1.

In [4]:
seed_concept_ids = set(seed_concepts_filtered['id:ID'])
has_concept_seed = has_concept[has_concept[':END_ID'].isin(seed_concept_ids)]

concept_count = has_concept_seed.groupby(':START_ID').size()

print("Distribuzione concetti seed per atto (cumulativa):")
print(f"  {'Soglia':>7}  {'Atti':>7}  {'% corpus':>8}")
print(f"  {'-'*7}  {'-'*7}  {'-'*8}")
max_show = min(int(concept_count.max()), 10) if len(concept_count) > 0 else 1
for thresh in range(1, max_show + 1):
    n = (concept_count >= thresh).sum()
    marker = "  <- MIN_SEED_CONCEPTS" if thresh == MIN_SEED_CONCEPTS else ""
    print(f"  >= {thresh:>4}  {n:>7,}  {n/len(nodes)*100:>7.1f}%{marker}")

print()

seed_work_ids         = set(concept_count[concept_count >= MIN_SEED_CONCEPTS].index)
seed_works_by_concept = nodes[nodes['id'].isin(seed_work_ids)].copy()

print(f"Atti L2 selezionati (MIN_SEED_CONCEPTS={MIN_SEED_CONCEPTS}): {len(seed_works_by_concept):,}")
print()
print("Per tipo:")
print(seed_works_by_concept['legal_type_normalized'].value_counts().to_string())

Distribuzione concetti seed per atto (cumulativa):
   Soglia     Atti  % corpus
  -------  -------  --------
  >=    1      559      0.9%
  >=    2       19      0.0%  <- MIN_SEED_CONCEPTS

Atti L2 selezionati (MIN_SEED_CONCEPTS=2): 19

Per tipo:
legal_type_normalized
Regulation    16
Decision       3


## 4. Costruzione Grafo Focale

Unisce L1 e L2, assegna `pipeline_level`, estrae archi interni.

In [5]:
all_seed_works = pd.concat([
    seed_nodes_known,
    seed_works_by_concept,
]).drop_duplicates(subset=['id'])

# Deduplica anche per celex — stesso atto può avere id diversi in nodes_light
before_celex = len(all_seed_works)
all_seed_works = all_seed_works.drop_duplicates(subset=['celex'], keep='first').reset_index(drop=True)
if before_celex > len(all_seed_works):
    print(f"Rimossi {before_celex - len(all_seed_works)} nodi con CELEX duplicato.")

seed_l1_ids = set(seed_nodes_known['id'])

focal_nodes = all_seed_works.copy()
focal_nodes['pipeline_level'] = focal_nodes['id'].apply(
    lambda x: 'L1_seed_manual' if x in seed_l1_ids else 'L2_seed_eurovoc'
)

print(f"=== GRAFO FOCALE ===")
print(f"Nodi: {len(focal_nodes):,} ({len(focal_nodes)/len(nodes)*100:.1f}% del corpus)")
print()
print("Per livello:")
print(focal_nodes['pipeline_level'].value_counts().to_string())
print()
print("Per tipo di atto:")
print(focal_nodes['legal_type_normalized'].value_counts().to_string())
print()
print("Per decade:")
print(focal_nodes['decade'].value_counts().sort_index().to_string())

=== GRAFO FOCALE ===
Nodi: 19 (0.0% del corpus)

Per livello:
pipeline_level
L2_seed_eurovoc    19

Per tipo di atto:
legal_type_normalized
Regulation    16
Decision       3

Per decade:
decade
2000.0    4
2010.0    6
2020.0    9


## 5. Estrazione Archi Interni

In [6]:
focal_ids     = set(focal_nodes['id'])
focal_celexes = set(focal_nodes['celex'].dropna())
celex_to_id   = focal_nodes.set_index('celex')['id'].to_dict()

def strip_corrigendum(val):
    if pd.isna(val): return val
    return re.sub(r'R\(\d+\)$', '', str(val))

focal_edges = edges[
    (edges[':START_ID'].isin(focal_ids) | edges[':START_ID'].isin(focal_celexes)) &
    (edges[':END_ID'].isin(focal_ids)   | edges[':END_ID'].isin(focal_celexes))
].copy()

focal_edges[':START_ID'] = focal_edges[':START_ID'].apply(lambda x: celex_to_id.get(x, x))
focal_edges[':END_ID']   = focal_edges[':END_ID'].apply(lambda x: celex_to_id.get(x, x))
focal_edges[':START_ID'] = focal_edges[':START_ID'].apply(strip_corrigendum)
focal_edges[':END_ID']   = focal_edges[':END_ID'].apply(strip_corrigendum)

print(f"Archi totali nel corpus:       {len(edges):>7,}")
print(f"Archi interni al grafo focale: {len(focal_edges):>7,}")
print()
print("Per tipo di relazione:")
print(focal_edges[':TYPE'].value_counts().to_string())

Archi totali nel corpus:       187,514
Archi interni al grafo focale:      22

Per tipo di relazione:
:TYPE
BASED_ON    11
AMENDS       9
CITES        2


## 6. Export

In [7]:
gephi_nodes = focal_nodes[[
    'id', 'celex', 'year_final', 'legal_type_normalized',
    'era', 'decade', 'pipeline_level'
]].copy()

gephi_nodes.rename(columns={
    'id':                    'Id',
    'celex':                 'Label',
    'year_final':            'Year',
    'legal_type_normalized': 'LegalType',
    'era':                   'Era',
    'decade':                'Decade',
    'pipeline_level':        'PipelineLevel',
}, inplace=True)

gephi_nodes['SeedConceptCount'] = gephi_nodes['Id'].apply(
    lambda x: int(concept_count.get(x, 0))
)

# In modalità theme: aggiungi similarity score medio dei concetti dell'atto
if MODE == "theme" and 'similarity' in seed_concepts_filtered.columns:
    concept_sim = seed_concepts_filtered.set_index('id:ID')['similarity'].to_dict()
    def mean_similarity(node_id):
        cids = has_concept_seed[has_concept_seed[':START_ID'] == node_id][':END_ID'].tolist()
        sims = [concept_sim[c] for c in cids if c in concept_sim]
        return round(float(np.mean(sims)), 4) if sims else 0.0
    gephi_nodes['AvgConceptSimilarity'] = gephi_nodes['Id'].apply(mean_similarity)

gephi_edges = focal_edges.rename(columns={
    ':START_ID': 'Source',
    ':END_ID':   'Target',
    ':TYPE':     'Type',
})

node_ids     = set(gephi_nodes['Id'])
dangling_src = ~gephi_edges['Source'].isin(node_ids)
dangling_tgt = ~gephi_edges['Target'].isin(node_ids)
print(f"Archi con Source non in nodi: {dangling_src.sum()}")
print(f"Archi con Target non in nodi: {dangling_tgt.sum()}")

gephi_nodes = gephi_nodes.drop_duplicates(subset='Label', keep='first').reset_index(drop=True)

gephi_nodes.to_csv(os.path.join(output_path, 'nodes_focal.csv'), index=False)
gephi_edges.to_csv(os.path.join(output_path, 'edges_focal.csv'), index=False)

print(f"\nFile salvati in {output_path}:")
print(f"  nodes_focal.csv  ({len(gephi_nodes):,} nodi, colonne: {list(gephi_nodes.columns)})")
print(f"  edges_focal.csv  ({len(gephi_edges):,} archi)")

Archi con Source non in nodi: 0
Archi con Target non in nodi: 0

File salvati in ..\data\output\fdi_screening:
  nodes_focal.csv  (19 nodi, colonne: ['Id', 'Label', 'Year', 'LegalType', 'Era', 'Decade', 'PipelineLevel', 'SeedConceptCount', 'AvgConceptSimilarity'])
  edges_focal.csv  (22 archi)


## 7. Riepilogo

In [8]:
print("=" * 60)
print("RIEPILOGO - ESTRAZIONE SOTTORETE")
print("=" * 60)
print(f"Modalità:             {MODE.upper()}")
print(f"Materia:              {MATERIA_NAME}")
print(f"Min concetti seed:    {MIN_SEED_CONCEPTS}")
print()

if MODE == "theme":
    print(f"Query:                '{THEME_QUERY}'")
    print(f"Top-K concetti:       {THEME_TOP_K}")
    print(f"Soglia similarità:    {THEME_MIN_SIMILARITY}")
    print(f"Concetti selezionati: {len(seed_concepts_filtered)}")
    if len(seed_concepts_filtered) > 0:
        best  = seed_concepts_filtered.nlargest(1, 'similarity').iloc[0]
        worst = seed_concepts_filtered.nsmallest(1, 'similarity').iloc[0]
        print(f"  Più simile:  [{best['similarity']:.3f}] {best['name']}")
        print(f"  Meno simile: [{worst['similarity']:.3f}] {worst['name']}")
elif MODE == "celex":
    print(f"CELEX seed:           {CELEX_SEEDS}")
    print(f"Concetti dai seed:    {len(seed_concepts_filtered)}")

print()
print("CORPUS DI PARTENZA")
print(f"  Atti totali:      {len(nodes):>7,}")
print(f"  Citazioni totali: {len(edges):>7,}")
print()
print("SOTTORETE ESTRATTA")
print(f"  Nodi L1:          {focal_nodes['pipeline_level'].eq('L1_seed_manual').sum():>5}")
print(f"  Nodi L2:          {focal_nodes['pipeline_level'].eq('L2_seed_eurovoc').sum():>5}")
print(f"  Nodi totali:      {len(focal_nodes):>5,} ({len(focal_nodes)/len(nodes)*100:.1f}% del corpus)")
print(f"  Archi interni:    {len(focal_edges):>5,}")

RIEPILOGO - ESTRAZIONE SOTTORETE
Modalità:             THEME
Materia:              fdi_screening
Min concetti seed:    2

Query:                'foreign direct investment screening'
Top-K concetti:       20
Soglia similarità:    0.25
Concetti selezionati: 20
  Più simile:  [0.660] foreign investment
  Meno simile: [0.430] export financing

CORPUS DI PARTENZA
  Atti totali:       62,244
  Citazioni totali: 187,514

SOTTORETE ESTRATTA
  Nodi L1:              0
  Nodi L2:             19
  Nodi totali:         19 (0.0% del corpus)
  Archi interni:       22
